# Tools
Tools are just the normal python functions used by LLM to extend their capabilities and provide them the ability to perform the tasks

there are 2 type of tools
1. prebuild tools: available with langchain_community.tools
2. custom tools

there are 3 ways to create a custom tool:

1. @tool decorator
2. StructuredTool function and Pydantic
3. BaseTool

### Note
- All the tools are not but the `Runnable`, we can invoke it by passing the args
- It is mandatory to pass the DOC String inside the function which helps LLM to understand the usability of tool. This Doc String will be treated as `description`
- every tool is having 4 things:

    1. ***name***: a name for the tool (by default function name)
    2. ***description***: doc string (by default LLM will use the doc string provided inside the func)
    3. ***args_schema***: parameters which we need to send to call the tool
    4. ***function***: normal python function which call when tool is `invoke()`


### Tool-Kit

A toolkit is a collection of related tools that serve a common purpose -  Packaged together for convenience and **reusability**


In LnagChain:
- a toolkit might be: `GoggleDRiveToolKit`
- and it can contain the following tools:

    - `GoogleDriveCreateFileTool`: upload a file
    - `GoogleDriveSearchFileTool`: search a file by name
    - `GoogleDriveReadFileTool`: Read contents of a file

# Tool Decorator

In [1]:
from langchain_community.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

/Users/bhavinp/workspace/langchain-practice/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
print(add_numbers)
"""
    Output:
        name='add_numbers' 
        description='Add two numbers together.' 
        args_schema=<class 'langchain_core.utils.pydantic.add_numbers'> 
        func=<function add_numbers at 0x12515b530>
"""

name='add_numbers' description='Add two numbers together.' args_schema=<class 'langchain_core.utils.pydantic.add_numbers'> func=<function add_numbers at 0x12515b530>


In [6]:
add_numbers.invoke({"a":2, "b":3})

5

# StructuredTool and PyDantic

In [12]:
from langchain_community.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyNumbersArgs(BaseModel):
    a: int = Field(required = True, description="The first number to multiply.")
    b: int = Field(required = True, description="The second number to multiply.")

def multiply_numbers(a: int, b: int) -> int:
    return a * b


# more verbose way to create a tool, but allows for more customization
# for example, we can add a description to the arguments, which will be used in the tool's documentation
multiply_tool = StructuredTool.from_function(
    func=multiply_numbers,
    name="multiply_numbers",
    description="Multiply two numbers together.",
    args_schema=MultiplyNumbersArgs
)

print(multiply_tool)
    

name='multiply_numbers' description='Multiply two numbers together.' args_schema=<class '__main__.MultiplyNumbersArgs'> func=<function multiply_numbers at 0x131646820>


/var/folders/nw/lx6yd1l55yg5sxwypptzzm_h0000gq/T/ipykernel_15748/659005935.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required = True, description="The first number to multiply.")
/var/folders/nw/lx6yd1l55yg5sxwypptzzm_h0000gq/T/ipykernel_15748/659005935.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required = True, description="The second number to multiply.")


In [26]:
multiply_tool.invoke({"a":2, "b":3})

6

In [ ]:
StructuredTool.

# BaseTool Class

In [20]:
from langchain_community.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

class MultiplyNumbersArgs(BaseModel):
    a: int = Field(required = True, description="The first number to multiply.")
    b: int = Field(required = True, description="The second number to multiply.")

class DivideNumbersTool(BaseTool):
    name: str = "divide_numbers"
    description: str = "Divide two numbers together."
    args_schema: Type[BaseTool] = MultiplyNumbersArgs

    # the _run method is where the logic of the tool goes, and is what will be called when the tool is invoked
    # the arguments to the _run method should match the fields of the args_schema
    def _run(self, a: int, b: int) -> int:
        return a / b
    
    

/var/folders/nw/lx6yd1l55yg5sxwypptzzm_h0000gq/T/ipykernel_15748/659415104.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required = True, description="The first number to multiply.")
/var/folders/nw/lx6yd1l55yg5sxwypptzzm_h0000gq/T/ipykernel_15748/659415104.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required = True, description="The second number to multiply.")


In [21]:
print(DivideNumbersTool)

<class '__main__.DivideNumbersTool'>


In [25]:
divide_tool = DivideNumbersTool()
divide_tool.invoke({"a":10, "b":2})

5.0

# Tool-kit

In [28]:
class MathToolkit:
    def get_tools(self):
        return [add_numbers, multiply_tool, divide_tool]
    
math_toolkit = MathToolkit()
for tool in math_toolkit.get_tools():
    print(tool.name, "=>", tool.description)

add_numbers => Add two numbers together.
multiply_numbers => Multiply two numbers together.
divide_numbers => Divide two numbers together.
